In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

# ---------------------------------------------------------
# 1. ГЕНЕРАЦІЯ РЕАЛІСТИЧНОГО ДАТАСЕТУ
# ---------------------------------------------------------
np.random.seed(42)  # Фіксація для відтворюваності
n_users = 10000

user_ids = [f"USR_{10000 + i}" for i in range(n_users)]
groups = np.random.choice(["A", "B"], size=n_users, p=[0.5, 0.5])

# Дати за останній місяць
start_date = pd.to_datetime("2026-10-01")
dates = start_date + pd.to_timedelta(
    np.random.randint(0, 30, size=n_users), unit="D"
)

devices = np.random.choice(
    ["Desktop", "Mobile", "Tablet"], size=n_users, p=[0.5, 0.4, 0.1]
)
sources = np.random.choice(
    ["Organic", "Paid", "Social", "Direct"],
    size=n_users,
    p=[0.35, 0.3, 0.2, 0.15],
)
countries = np.random.choice(
    ["UA", "PL", "DE", "US"], size=n_users, p=[0.4, 0.25, 0.2, 0.15]
)
sessions = np.random.poisson(lam=3, size=n_users) + 1

# Моделюємо ефект A/B тесту (Варіант B має вищу конверсію та трохи вищий чек)
add_to_cart_prob = np.where(groups == "A", 0.25, 0.29)
purchased_prob = np.where(groups == "A", 0.10, 0.125)  # +2.5% Absolute Lift в B

added_to_cart = np.random.binomial(1, add_to_cart_prob)
# Купити можуть лише ті, хто додав у кошик (чи логіка)
purchased = np.where(
    added_to_cart == 1, np.random.binomial(1, purchased_prob / add_to_cart_prob), 0
)

# Розрахунок доходу (Revenue)
base_revenue = np.random.gamma(shape=3, scale=25, size=n_users)  # Log-normal/Gamma distribution
# У групі B чек трохи вищий
revenue = np.where(purchased == 1, np.round(base_revenue * np.where(groups == "B", 1.05, 1.0), 2), 0.0)

# Формуємо DataFrame сирих даних
df_raw = pd.DataFrame(
    {
        "user_id": user_ids,
        "experiment_group": groups,
        "experiment_date": dates,
        "device": devices,
        "traffic_source": sources,
        "sessions": sessions,
        "added_to_cart": added_to_cart,
        "purchased": purchased,
        "revenue": revenue,
        "country": countries,
    }
)

# Зберігаємо сирий датасет
df_raw.to_csv("ab_test_raw_data.csv", index=False)
print("✅ Сирий датасет успішно збережено в 'ab_test_raw_data.csv'")

# ---------------------------------------------------------
# 2. СТАТИСТИЧНИЙ АНАЛІЗ ТА ОБЧИСЛЕННЯ МЕТРИК
# ---------------------------------------------------------
group_a = df_raw[df_raw["experiment_group"] == "A"]
group_b = df_raw[df_raw["experiment_group"] == "B"]

n_a, n_b = len(group_a), len(group_b)
buyers_a, buyers_b = group_a["purchased"].sum(), group_b["purchased"].sum()
cart_a, cart_b = group_a["added_to_cart"].sum(), group_b["added_to_cart"].sum()

# --- МЕТРИКИ ---
# 1. Add-to-Cart Rate
atc_rate_a = cart_a / n_a
atc_rate_b = cart_b / n_b

# 2. Conversion Rate (CR)
cr_a = buyers_a / n_a
cr_b = buyers_b / n_b

# 3. Revenue per User (RPU)
rpu_a = group_a["revenue"].sum() / n_a
rpu_b = group_b["revenue"].sum() / n_b

# 4. Average Order Value (AOV)
aov_a = group_a[group_a["purchased"] == 1]["revenue"].mean()
aov_b = group_b[group_b["purchased"] == 1]["revenue"].mean()

# --- СТАТИСТИЧНІ ТЕСТИ ---
# Z-test для Conversion Rate
z_stat_cr, p_val_cr = proportions_ztest([buyers_b, buyers_a], [n_b, n_a])

# Z-test для Add-to-Cart Rate
z_stat_atc, p_val_atc = proportions_ztest([cart_b, cart_a], [n_b, n_a])

# Welch's t-test для Revenue per User
t_stat_rpu, p_val_rpu = stats.ttest_ind(
    group_b["revenue"], group_a["revenue"], equal_var=False
)

# Welch's t-test для AOV (лише для покупців)
t_stat_aov, p_val_aov = stats.ttest_ind(
    group_b[group_b["purchased"] == 1]["revenue"],
    group_a[group_a["purchased"] == 1]["revenue"],
    equal_var=False,
)

# --- ПІДГОТОВКА ЗВЕДЕНОЇ ТАБЛИЦІ МЕТРИК ДЛЯ TABLEAU ---
metrics_summary = [
    {
        "Metric": "Conversion Rate",
        "Group_A": round(cr_a, 4),
        "Group_B": round(cr_b, 4),
        "Absolute_Lift": round(cr_b - cr_a, 4),
        "Relative_Lift_%": round(((cr_b - cr_a) / cr_a) * 100, 2),
        "P_Value": round(p_val_cr, 5),
        "Is_Stat_Significant": p_val_cr < 0.05,
    },
    {
        "Metric": "Add-to-Cart Rate",
        "Group_A": round(atc_rate_a, 4),
        "Group_B": round(atc_rate_b, 4),
        "Absolute_Lift": round(atc_rate_b - atc_rate_a, 4),
        "Relative_Lift_%": round(((atc_rate_b - atc_rate_a) / atc_rate_a) * 100, 2),
        "P_Value": round(p_val_atc, 5),
        "Is_Stat_Significant": p_val_atc < 0.05,
    },
    {
        "Metric": "Revenue per User (RPU)",
        "Group_A": round(rpu_a, 2),
        "Group_B": round(rpu_b, 2),
        "Absolute_Lift": round(rpu_b - rpu_a, 2),
        "Relative_Lift_%": round(((rpu_b - rpu_a) / rpu_a) * 100, 2),
        "P_Value": round(p_val_rpu, 5),
        "Is_Stat_Significant": p_val_rpu < 0.05,
    },
    {
        "Metric": "Average Order Value (AOV)",
        "Group_A": round(aov_a, 2),
        "Group_B": round(aov_b, 2),
        "Absolute_Lift": round(aov_b - aov_a, 2),
        "Relative_Lift_%": round(((aov_b - aov_a) / aov_a) * 100, 2),
        "P_Value": round(p_val_aov, 5),
        "Is_Stat_Significant": p_val_aov < 0.05,
    },
]

df_summary = pd.DataFrame(metrics_summary)
df_summary.to_csv("ab_test_stats_summary.csv", index=False)
print("✅ Статистичний звіт збережено в 'ab_test_stats_summary.csv'")

✅ Сирий датасет успішно збережено в 'ab_test_raw_data.csv'
✅ Статистичний звіт збережено в 'ab_test_stats_summary.csv'
